## Pacote e bibliotecas necessárias.

In [1]:
# Instalar ou atualizar biblioteca necessária.
!pip install -q -U datasets

# Importar bibliotecas necessárias
import pandas as pd
import os
import re
import gc
import time
from openai import OpenAI
from google.colab import userdata
from datasets import load_dataset

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.1 MB/s eta 0:00:00


Funções que povoam os Dataframes com as questões e respostas disponíveis no github.

In [36]:
# Povoar dataframe de questões do hugging face.
def load_questions(hugging_face_id):
  # Carrega o dataset (geralmente ele vem dividido em 'train', 'test', etc.)
  ds = load_dataset(hugging_face_id)

  # Converte uma partição específica 'train' para um dataframe.
  df = pd.DataFrame(ds['train'])

  # Inserir uma coluna para enumerar as questões, com contagem a partir do número 1, uma vez que a contagem de linhas do python é a partir do 0.
  df.insert(0, 'num', range(1, len(df)+1))

  # retorno da Dataframe.
  return df

# Dataframe com um subconjunto das perguntas e linhas guias.
# O índice do Dataframe começa do 0 (zero), portanto, a linha 1 é a 0, a 2 é a 1, etc.
# iloc seleciona um intervalo fechado à esquerda e aberto à direita
def load_my_questions(question_min, question_max, columns=None):
  # Remover 1 und dando o descontando, uma vez que o valor passado é o número da questão para o ser humano
  # e não a contagem do python.
  question_min -= 1
  df_my_close_questions = df_close_questions.iloc[question_min:question_max].copy()
  if columns is not None:
    return df_my_close_questions[columns]
  return df_my_close_questions

# Excluir Dataframes desnecessários.
def delete_obsoletes_objects():
  # Excluir Dataframes que não são mais necessários, apenas se existirem no escopo global.
  if 'df_close_questions' in globals():
    del df_questions
  if 'df' in globals():
    del df_my_questions
  if 'ds' in globals():
    del ds
  gc.collect()

# Função para converter um dataframe em jsonl e salvá-lo com arquivo no colab.
def save_df_to_jsonl(df: pd.DataFrame, filename: str):
  try:
    # Converte o DataFrame para JSONL
    # orient='records' serializa cada linha como um objeto JSON
    # lines=True garante que cada objeto JSON esteja em uma nova linha (formato JSONL)
    # json_format='plain' evita o escape de caracteres não-ASCII, preservando a acentuação.
    #jsonl_output = df.to_json(orient='records', lines=True, json_format='plain')

    # Exportando o dataframe para JSONL de forma correta
    df.to_json(
        filename,
        orient='records',
        lines=True,
        force_ascii=False  # <--- Não converter para ascii.
    )
    print(f"DataFrame salvo com sucesso em {filename} no formato JSONL.")
  except Exception as e:
    print(f"Ocorreu um erro ao salvar o DataFrame em JSONL: {e}")

## Funçõoes para IA

In [ ]:
def markdown_prepare(papel, categoria, contexto, dificuldade, instrucao):
  # Preparação do prompt em Markdown
  prompt_usuario = f"""
  ### Papel:
  {papel}

  ### Área de atuação:
  {categoria}

  ### Contexto:
  {contexto}

  ### Nível de Dificuldade:
  Com valores de 1 a 4, onde 1 indica o mais simples e 4 o mais complexo.
  {dificuldade}

  ### Instrução:
  {instrucao}
  """

  return prompt_usuario

def question_submit(client_ai, model_id, prompt):
  # Realizar consulta a IA.
  completion = client_ai.chat.completions.create(
      model= model_id,
      messages=[
        # Por questões de sintaxes informo a role, pois é um campo obrigatório,
        # porém o conteúdo é somente o do Markdown.
        {"role": "user", "content": prompt}
      ],
      temperature=0.1 # Conservador.
  )
  response = completion.choices[0].message.content
  return responses

Todas as perguntas objetivas disponíveis.

In [42]:
os.environ["HF_TOKEN"] = userdata.get('hugging_colab')

hugging_face_id = 'eduagarcia/oab_exams'
df_close_questions = load_questions(hugging_face_id)

## Apenas as perguntas objetivas que vou analisar.

In [46]:
# Meu sub-grupo de questões e respostas.
# Subconjunto das questões de 319 a 424.
# E já seleciono as colunas que vou usar.
question_min = 319
question_max = 424
columns=['num', 'question_number', 'exam_year', 'question_type', 'question', 'choices', 'answerKey']
df_my_close_questions = load_my_questions(question_min, question_max, columns)

,num,question_number,exam_year,question_type,question,choices,answerKey
318,319,20,2011,CONSTITUTIONAL,Em relação ao controle de constitucionalidade ...,{'text': ['Compete aos Estados a instituição d...,C
319,320,21,2011,CONSTITUTIONAL,As alternativas a seguir apontam diferenças en...,{'text': ['Rol de legitimados para a propositu...,A
320,321,22,2011,CONSTITUTIONAL,A respeito da distribuição de competências ado...,{'text': ['A competência material da União pod...,B
321,322,23,2011,CONSTITUTIONAL,Os direitos políticos não podem ser cassados. ...,{'text': ['condenação cível sem trânsito em ju...,D
322,323,24,2011,CONSTITUTIONAL,Considere a hipótese de Deputado Federal que c...,{'text': ['a Câmara dos Deputados pode sustar ...,A


Função para definir o nível de complexidade das questões.

In [47]:
# Classificar os níveis das perguntas.
def classify_difficulty_level():
  # Recuperar a chave da API de forma segura, armazenada no Secrets do Google Colab.
  # Tal chave previamente criada no console do Groq, site: consele.groq.com.
  groq_api_key = userdata.get('groq_api_key')

  # Inicializar o cliente da API.
  client_ai = OpenAI(
      base_url="https://api.groq.com/openai/v1",
      api_key=groq_api_key
  )

  # Modelo llama 3 de 70 bilhões de parâmetros.
  model_id = 'llama-3.3-70b-versatile'

  # Inicializar a coluna 'level'.
  df_my_close_questions['level'] = None

  # Repetição para percorrer todo Dataframe.
  for index, row in df_my_close_questions.iterrows():
      # preenchimento dos parâmetros da pergunta, com base na linha corrente.
      numero = row['num']
      contexto = row['question']
      opcoes = row['choices']

      # Preparação do prompt em Markdown.
      prompt_usuario = f"""
      Analise a complexidade jurídica da seguinte questão e atribua um nível de dificuldade de 1 a 4, onde:
      1: Estagiário (leis e conceitos jurídicos muito básicos, aplicação direta de uma única regra).
      2: Analista (leis e conceitos jurídicos moderados, aplicação de 1-2 conceitos principais, requer identificação de normas).
      3: Juiz (leis e conceitos jurídicos complexos, múltiplos conceitos interconectados, exige análise aprofundada, interpretação de leis e jurisprudência).
      4: Ministro (leis e conceitos jurídicos muito complexos e intrincados, temas controversos, requer alta interpretação constitucional ou legal, análise de precedentes e doutrina avançada).

      ### Contexto:
      {contexto}

      ### Opções de resposta:
      {opcoes}

      Responda apenas com o número do nível de dificuldade (1, 2, 3 ou 4), sem texto adicional.
      """
      completion = client_ai.chat.completions.create(
          model= model_id,
          messages=[
            # Por questões de sintaxes informo a role, pois é um campo obrigatório,
            # porém o conteúdo é somente o do Markdown.
            {"role": "user", "content": prompt_usuario}
          ],
          temperature=0.1 # Conservador.
      )
      response = completion.choices[0].message.content

  # Armazenar o resultado diretamente no DataFrame df_my_questions
  df_my_close_questions.loc[index, 'level'] = response

  # Fechar conexão com a IA (somente se você a usou ativamente)
  client_ai.close()

Chamar a função que adiciona o nível de difuculdade no dataframe df_my_questions.

In [48]:
classify_difficulty_level()

Liberar espeço não necessário em memória.

In [ ]:
delete_obsoletes_objects()

In [ ]:
def prepare_markdown_dynamic(data):
  """
  Gera um texto Markdown dinamicamente a partir de um dicionário (ou similar).

  Args:
    data (dict or list of tuples): Um dicionário onde as chaves serão os títulos
                                    e os valores o conteúdo, ou uma lista de tuplas
                                    (chave, valor).

  Returns:
    str: O texto formatado em Markdown.
  """
  markdown_text = []
  if isinstance(data, dict):
    items = data.items()
  elif isinstance(data, (list, tuple)) and all(isinstance(item, (list, tuple)) and len(item) == 2 for item in data):
    items = data
  else:
    raise TypeError("O argumento 'data' deve ser um dicionário ou uma lista/tupla de tuplas (chave, valor).")

  for key, value in items:
    markdown_text.append(f"### {key}")
    markdown_text.append(f"{value}\n") # Adiciona uma quebra de linha extra para espaçamento

  return "\n".join(markdown_text).strip()

# Exemplo de uso com um dicionário
# dados_exemplo = {
#     "Persona": "Um estudante de direito que busca questões para praticar",
#     "Área de atuação": "Direito Constitucional",
#     "Contexto": "Elaborar perguntas de múltipla escolha sobre competências legislativas",
#     "Nível de Dificuldade": "3"
# }
# print(prepare_markdown_dynamic(dados_exemplo))

# Exemplo de uso com uma lista de tuplas
# dados_lista = [
#     ("Assunto", "Direitos Fundamentais"),
#     ("Tipo", "Questões abertas"),
#     ("Observação", "Focar em casos práticos")
# ]
# print(prepare_markdown_dynamic(dados_lista))